# Korean Variety Show → English Subtitles

Automates the AI-able parts of the pipeline: extract audio → transcribe (Korean) → rough translate (English).

**This notebook does NOT do the manual steps** (tone/translation cleanup, timing fixes, burned-in on-screen text). Open the output `.srt` in Aegisub or Subtitle Edit afterward for that.

Works on Google Colab (free T4 GPU) or locally if you have a GPU.

In [ ]:
# Install dependencies
!pip install -q faster-whisper deep-translator ffmpeg-python
# Colab usually has ffmpeg pre-installed; uncomment if needed locally:
# !apt-get -y install ffmpeg -qq

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## Config
Set the path to your video and your preferred model size below.

In [ ]:
# Path to your video file.
# - On Colab: upload to Google Drive first, e.g. "/content/drive/MyDrive/videos/episode1.mp4"
# - Locally: any path on disk, e.g. "./episode1.mp4"
VIDEO_PATH = "/content/drive/MyDrive/videos/episode1.mp4"

# Where audio + subtitle outputs get written
OUTPUT_DIR = "/content/drive/MyDrive/subtitles" if IN_COLAB else "./output"

WHISPER_MODEL_SIZE = "large-v3"   # tiny / base / small / medium / large-v3
SOURCE_LANGUAGE = "ko"
TARGET_LANGUAGE = "en"
DEVICE = "cuda"          # "cuda" if you have a GPU (Colab free T4 works), else "cpu"
COMPUTE_TYPE = "int8_float16"   # int8 variants keep VRAM usage low

## Step 1: Extract audio

In [ ]:
import os
import ffmpeg

os.makedirs(OUTPUT_DIR, exist_ok=True)
basename = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
AUDIO_PATH = os.path.join(OUTPUT_DIR, f"{basename}.wav")

(
    ffmpeg
    .input(VIDEO_PATH)
    .output(AUDIO_PATH, ac=1, ar=16000, vn=None)
    .overwrite_output()
    .run(quiet=True)
)
print(f"Audio extracted to {AUDIO_PATH}")

## Step 2: Transcribe (Korean)
Uses `task="transcribe"`, not `"translate"` — Whisper's built-in translation is unreliable for casual/idiomatic speech. We translate separately in Step 3.

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(WHISPER_MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

segments, info = model.transcribe(
    AUDIO_PATH,
    language=SOURCE_LANGUAGE,
    task="transcribe",
    vad_filter=True,   # skips silence/non-speech, helps with long files
)

segments = list(segments)  # materialize the generator so we can reuse it below
print(f"Detected language: {info.language} (confidence {info.language_probability:.2f})")
print(f"Transcribed {len(segments)} segments")

## Step 3: Write Korean subtitles + rough English translation

In [ ]:
def format_timestamp(seconds: float) -> str:
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

KOREAN_SRT_PATH = os.path.join(OUTPUT_DIR, f"{basename}_ko.srt")

with open(KOREAN_SRT_PATH, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, start=1):
        f.write(f"{i}\n")
        f.write(f"{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}\n")
        f.write(f"{seg.text.strip()}\n\n")

print(f"Korean subtitles written to {KOREAN_SRT_PATH}")

In [ ]:
from deep_translator import GoogleTranslator
# Swap GoogleTranslator for DeeplTranslator(api_key=...) if you have a DeepL key — generally better quality.

translator = GoogleTranslator(source=SOURCE_LANGUAGE, target=TARGET_LANGUAGE)
ENGLISH_SRT_PATH = os.path.join(OUTPUT_DIR, f"{basename}_en.srt")

with open(ENGLISH_SRT_PATH, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, start=1):
        try:
            translated = translator.translate(seg.text.strip())
        except Exception as e:
            translated = seg.text.strip()  # fall back to Korean if translation fails
            print(f"Translation failed for segment {i}: {e}")
        f.write(f"{i}\n")
        f.write(f"{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}\n")
        f.write(f"{translated}\n\n")

print(f"Rough English subtitles written to {ENGLISH_SRT_PATH}")

## Next steps (manual — not automatable)

1. Download/locate `{basename}_en.srt` from `OUTPUT_DIR`.
2. Open it alongside the video in **Aegisub** or **Subtitle Edit**.
3. Fix mistranslations, tone/jokes, and timing drift.
4. Manually add subtitle lines for any **burned-in on-screen text** (reaction captions, name tags) — the model never saw these, since they're graphics, not audio.
5. Export your final `.srt`/`.ass`.